# Лабораторна 1 — дослід: каталог навчальних відео
Доопрацьований дослід від 2026-11-01. Початкові результати є в Git; цей прогін не видається за дослід до першої реалізації.
Поля: title, topic, duration_minutes. Очікування нижче зафіксовані до викликів. Неоднозначні/неповні дані → уточнення (None). Схема не доводить відповідність тексту.
Запуск із кореня проєкту або notebooks. GROQ_API_KEY береться з .env, не друкується. Реальні виклики виконуються лише тут і на live-сервері. Підготовлені пропозиції використовуються окремо для контрактних тестів.


In [1]:
import json, re, os
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq
ROOT = Path.cwd() if (Path.cwd()/"app").exists() else Path.cwd().parent
load_dotenv(ROOT/".env")
cases = [{'id': 'normal', 'text': 'Відео Python, тема цикли, тривалість 15 хвилин', 'expected': {'title': 'Python', 'topic': 'цикли', 'duration_minutes': 15}}, {'id': 'lesson', 'text': 'Урок HTML; тема форми; тривалість 20 хв', 'expected': {'title': 'HTML', 'topic': 'форми', 'duration_minutes': 20}}, {'id': 'label', 'text': 'Назва: SQL; тема: бази даних; тривалість: 30 хв', 'expected': {'title': 'SQL', 'topic': 'бази даних', 'duration_minutes': 30}}, {'id': 'spaces', 'text': ' Відео CSS , тема стилі , тривалість 10 хвилин ', 'expected': {'title': 'CSS', 'topic': 'стилі', 'duration_minutes': 10}}, {'id': 'case', 'text': 'відео Git, ТЕМА гілки, ТРИВАЛІСТЬ 12 хв', 'expected': {'title': 'Git', 'topic': 'гілки', 'duration_minutes': 12}}, {'id': 'missing_duration', 'text': 'Відео Python, тема цикли', 'expected': None}, {'id': 'ambiguous', 'text': 'Відео Python, тема цикли, тривалість 15 або 20 хвилин', 'expected': None}, {'id': 'zero', 'text': 'Відео Python, тема цикли, тривалість 0 хвилин', 'expected': None}, {'id': 'missing_topic', 'text': 'Відео Python, тривалість 15 хвилин', 'expected': None}, {'id': 'paraphrase', 'text': 'Ролик про цикли з назвою Python займає 15 хвилин', 'expected': {'title': 'Python', 'topic': 'цикли', 'duration_minutes': 15}}]
print("Зафіксовано випадків:", len(cases))

Зафіксовано випадків: 10


In [2]:

def valid(value):
    return (isinstance(value,dict) and set(value)=={'title','topic','duration_minutes'}
            and all(isinstance(value[k],str) and 0<len(value[k].strip())<=200 for k in ['title','topic'])
            and type(value['duration_minutes']) is int and 1<=value['duration_minutes']<=1440)

def baseline(text):
    m=re.fullmatch(r'\s*(?:Відео\s+|Урок\s+|Назва\s*:\s*)([^,;\n]+?)\s*[,;]\s*тема\s*:?\s+([^,;\n]+?)\s*[,;]\s*тривалість\s*:?\s+(\d+)\s*(?:хвилин|хв)\s*\.?\s*',text,re.I)
    if not m: return None
    value={'title':m[1].strip(),'topic':m[2].strip(),'duration_minutes':int(m[3])}
    return value if valid(value) else None

def accept(proposal, explicit):
    return valid(proposal) and explicit is not None and proposal==explicit

# Незалежні підготовлені відповіді; це не результати провайдера.
reference={'title':'Python','topic':'цикли','duration_minutes':15}
checks=[accept(reference, reference),
        not accept({**reference,'duration_minutes':21},reference),
        not accept({**reference,'title':'Java'},reference),
        not accept({**reference,'topic':'масиви'},reference),
        not valid({**reference,'duration_minutes':True}),
        not valid({**reference,'duration_minutes':'15'}),
        not valid({**reference,'duration_minutes':0}),
        not valid({**reference,'extra':1}),
        not accept(None,reference), not accept(reference,None)]
assert all(checks)
print(f'Детерміновані перевірки контракту: {sum(checks)}/{len(checks)} passed')
# Перевірка чутливості: навмисне неправильне число має провалити позитивне очікування.
try:
    assert accept({**reference,'duration_minutes':21},reference)
except AssertionError:
    print('Чутливість: неправильні 21 замість 15 виявлено (очікувана AssertionError).')
else:
    raise AssertionError('Перевірка нечутлива до помилки')


Детерміновані перевірки контракту: 10/10 passed
Чутливість: неправильні 21 замість 15 виявлено (очікувана AssertionError).


## Контрольований експеримент A/B
Для обох інструкцій однакові 10 входів, очікування, модель, temperature=0 і спосіб оцінювання (точна рівність). B відрізняється лише уточненням межі назви: вступний маркер не є її частиною. Помилки моделі не спричиняють зміни очікувань. Перефразування навмисно показує межу baseline.


In [3]:

prompt_a='Витягни дані про навчальне відео. Поверни тільки JSON з title, topic, duration_minutes або null за відсутніх чи неоднозначних даних. Тривалість — додатне ціле до 1440. Назву й тему збережи дослівно; не вигадуй.'
prompt_b=prompt_a+' Вступні маркери «Відео», «Урок», «Назва:» не включай в title.'
key=os.getenv('GROQ_API_KEY')
model=os.getenv('GROQ_MODEL','openai/gpt-oss-20b')
print('Провайдер: Groq; модель:', model)
rows=[]
client=Groq(api_key=key,timeout=15,max_retries=0) if key else None
try:
    for case in cases:
        row={'case':case['id'],'expected':case['expected'],'rules':baseline(case['text'])}
        row['rules_status']='passed' if row['rules']==row['expected'] else 'failed'
        for version,prompt in [('A',prompt_a),('B',prompt_b)]:
            if client is None:
                row[version]={'status':'not_run','reason':'GROQ_API_KEY відсутній'}
                continue
            try:
                response=client.chat.completions.create(model=model,temperature=0,max_tokens=512,messages=[{'role':'system','content':prompt},{'role':'user','content':case['text']}])
            except Exception as exc:
                row[version]={'status':'not_run','reason':type(exc).__name__}
                continue
            raw=response.choices[0].message.content
            try:
                value=json.loads(raw)
                passed=(value is None and case['expected'] is None) or (valid(value) and value==case['expected'])
                row[version]={'status':'passed' if passed else 'failed','value':value}
            except (ValueError,TypeError):
                row[version]={'status':'failed','reason':'invalid JSON','raw':raw}
        rows.append(row)
        print(json.dumps(row,ensure_ascii=False))
finally:
    if client: client.close()


Провайдер: Groq; модель: openai/gpt-oss-20b


{"case": "normal", "expected": {"title": "Python", "topic": "цикли", "duration_minutes": 15}, "rules": {"title": "Python", "topic": "цикли", "duration_minutes": 15}, "rules_status": "passed", "A": {"status": "failed", "value": {"title": "Відео Python", "topic": "цикли", "duration_minutes": 15}}, "B": {"status": "passed", "value": {"title": "Python", "topic": "цикли", "duration_minutes": 15}}}


{"case": "lesson", "expected": {"title": "HTML", "topic": "форми", "duration_minutes": 20}, "rules": {"title": "HTML", "topic": "форми", "duration_minutes": 20}, "rules_status": "passed", "A": {"status": "failed", "value": {"title": "Урок HTML", "topic": "тема форми", "duration_minutes": 20}}, "B": {"status": "passed", "value": {"title": "HTML", "topic": "форми", "duration_minutes": 20}}}


{"case": "label", "expected": {"title": "SQL", "topic": "бази даних", "duration_minutes": 30}, "rules": {"title": "SQL", "topic": "бази даних", "duration_minutes": 30}, "rules_status": "passed", "A": {"status": "passed", "value": {"title": "SQL", "topic": "бази даних", "duration_minutes": 30}}, "B": {"status": "passed", "value": {"title": "SQL", "topic": "бази даних", "duration_minutes": 30}}}


{"case": "spaces", "expected": {"title": "CSS", "topic": "стилі", "duration_minutes": 10}, "rules": {"title": "CSS", "topic": "стилі", "duration_minutes": 10}, "rules_status": "passed", "A": {"status": "failed", "value": {"title": "Відео CSS", "topic": "стилі", "duration_minutes": 10}}, "B": {"status": "passed", "value": {"title": "CSS", "topic": "стилі", "duration_minutes": 10}}}


{"case": "case", "expected": {"title": "Git", "topic": "гілки", "duration_minutes": 12}, "rules": {"title": "Git", "topic": "гілки", "duration_minutes": 12}, "rules_status": "passed", "A": {"status": "failed", "value": {"title": "відео Git", "topic": "ТЕМА гілки", "duration_minutes": 12}}, "B": {"status": "failed", "reason": "invalid JSON", "raw": ""}}


{"case": "missing_duration", "expected": null, "rules": null, "rules_status": "passed", "A": {"status": "failed", "value": {"title": "Відео Python", "topic": "тема цикли", "duration_minutes": null}}, "B": {"status": "failed", "reason": "invalid JSON", "raw": "```json\n{\n  \"title\": \"Python\",\n  \"topic\": \"цикли\",\n  \"duration_minutes\": null\n}\n```"}}


{"case": "ambiguous", "expected": null, "rules": null, "rules_status": "passed", "A": {"status": "failed", "value": {"title": null, "topic": "цикли", "duration_minutes": null}}, "B": {"status": "failed", "value": {"title": "Python", "topic": "цикли", "duration_minutes": null}}}


{"case": "zero", "expected": null, "rules": null, "rules_status": "passed", "A": {"status": "failed", "value": {"title": "Відео Python", "topic": "тема цикли", "duration_minutes": null}}, "B": {"status": "failed", "value": {"title": "Python", "topic": "цикли", "duration_minutes": null}}}


{"case": "missing_topic", "expected": null, "rules": null, "rules_status": "passed", "A": {"status": "failed", "value": {"title": "Відео Python", "topic": null, "duration_minutes": 15}}, "B": {"status": "failed", "value": {"title": "Python", "topic": null, "duration_minutes": 15}}}


{"case": "paraphrase", "expected": {"title": "Python", "topic": "цикли", "duration_minutes": 15}, "rules": null, "rules_status": "failed", "A": {"status": "passed", "value": {"title": "Python", "topic": "цикли", "duration_minutes": 15}}, "B": {"status": "passed", "value": {"title": "Python", "topic": "цикли", "duration_minutes": 15}}}


In [4]:

from IPython.display import Markdown, display
table='|Випадок|Правила|A (live)|B (live)|\n|---|---|---|---|\n'
for row in rows:
    table+=f"|{row['case']}|{row['rules_status']}|{row['A']['status']}|{row['B']['status']}|\n"
display(Markdown(table))
summary={'rules':{s:sum(r['rules_status']==s for r in rows) for s in ['passed','failed']}}
for v in ['A','B']:
    summary[v]={s:sum(r[v]['status']==s for r in rows) for s in ['passed','failed','not_run']}
print(json.dumps(summary,ensure_ascii=False,indent=2))
(ROOT/'reports').mkdir(exist_ok=True)
(ROOT/'reports'/'experiment.json').write_text(json.dumps({'cases':rows,'summary':summary,'contract_checks':10,'model':model},ensure_ascii=False,indent=2),encoding='utf-8')


|Випадок|Правила|A (live)|B (live)|
|---|---|---|---|
|normal|passed|failed|passed|
|lesson|passed|failed|passed|
|label|passed|passed|passed|
|spaces|passed|failed|passed|
|case|passed|failed|failed|
|missing_duration|passed|failed|failed|
|ambiguous|passed|failed|failed|
|zero|passed|failed|failed|
|missing_topic|passed|failed|failed|
|paraphrase|failed|passed|passed|


{
  "rules": {
    "passed": 9,
    "failed": 1
  },
  "A": {
    "passed": 2,
    "failed": 8,
    "not_run": 0
  },
  "B": {
    "passed": 5,
    "failed": 5,
    "not_run": 0
  }
}


5818

## Висновок
Правила достатні для явного шаблону й навмисно не розуміють вільне перефразування. LLM може розпізнати таке повідомлення, але точні успіхи/помилки конкретного прогону показано вище, без узагальнення на всі запити. Продукт приймає лише явний шаблон: це дозволяє незалежно звірити всі поля й безпечно відмовити в неоднозначному випадку. Для такого вузького продукту LLM не є необхідною; вона включена як навчальний компонент. До сервісу перенесено перевірку типів, діапазону, відповідності тексту та відмову без збереження. Контрактні тести — доказ логіки; live-таблиця — оцінювання моделі, not_run не є успіхом.
